# Importing Required Libraries

In [1]:
import requests
import time
import pandas as pd
import re
from urllib.parse import quote_plus
from googlesearch import search
import os
from dotenv import load_dotenv

In [3]:
import csv

# Environment Setup & Initialization

In [2]:
load_dotenv()

# Récupérer la clé
SERPAPI_KEY = os.getenv("SERPAPI_KEY")
HUNTER_KEY = os.getenv("HUNTER_KEY")
companies = ["Deepgram",
    "Runway",
    "ConsenSys",
    "Replika",
    "Cohere"]
# Keywords to find relevant contacts
keywords = ["marketing", "innovation", "ai", "growth", "business development"]

# Extract Company Domain via SerpAPI

In [78]:
def company_to_domain(company):
    """Get company domain from SerpAPI Google search"""
    url = "https://serpapi.com/search"
    params = {"q": company, "engine": "google", "api_key": SERPAPI_KEY}
    r = requests.get(url, params=params)
    if r.status_code != 200:
        print(f"[SerpAPI error] {company}: {r.text}")
        return None, None
    results = r.json().get("organic_results", [])
    if not results:
        return None, None
    link = results[0].get("link", "")
    snippet = results[0].get("snippet", "")
    domain = link.split("/")[2].replace("www.", "") if link else None
    return domain, snippet  # snippet used as company field



# Retrieve Emails from Hunter.io

In [79]:
def hunter_domain_search(domain):
    """Get up to 10 emails from Hunter free plan"""
    url = "https://api.hunter.io/v2/domain-search"
    params = {"domain": domain, "api_key": HUNTER_KEY, "limit": 10}
    r = requests.get(url, params=params)
    if r.status_code != 200:
        print(f"[Hunter error] {domain}: {r.text}")
        return []
    return r.json().get("data", {}).get("emails", [])

# Identify Relevant Contact by Role

In [80]:
def find_relevant_contact(emails):
    """Filter emails for relevant roles"""
    for e in emails:
        pos = (e.get("position") or "").lower()
        if any(k.lower() in pos for k in keywords):
            return {
                "Full name": f"{e.get('first_name','')} {e.get('last_name','')}".strip(),
                "Job title": e.get("position"),
                "Email": e.get("value"),
                "LinkedIn URL": e.get("linkedin") or ""
            }
    return None


# Main Loop: Build Contact List for Target Companies

In [84]:
contacts_list = []

for company in companies:
    domain, company_field  = company_to_domain(company)
    if not domain:
        contacts_list.append({
            "Full name": "",
            "Job title": "",
            "Company name": company,
            "Company domain": "",
            "Company_Field": "",
            "Email": "",
            "LinkedIn URL": ""
        })
        continue

    print(f"[{company}] Domain: {domain}")
    time.sleep(1)  # avoid SerpAPI rate limits

    emails = hunter_domain_search(domain)
    contact = find_relevant_contact(emails)

    if not contact:
        contact = {"Full name": "", "Email": "", "Job title": "", "LinkedIn URL": ""}

    contact.update({"Company name": company, "Company domain": company_field}) #"Company domain": domain
    contacts_list.append(contact)

[Deepgram] Domain: deepgram.com
[Runway] Domain: runwayml.com
[ConsenSys] Domain: consensys.io
[Replika] Domain: replika.com
[Cohere] Domain: cohere.com


# Export Contacts to CSV

In [85]:
# ---------------- EXPORT CSV ----------------
df = pd.DataFrame(contacts_list)
df.to_csv("ai_event_contacts.csv", index=False)
print("Saved 5 contacts (1 per company) -> ai_event_contacts.csv")

Saved 5 contacts (1 per company) -> ai_event_contacts.csv


In [86]:
df

,Full name,Job title,Email,LinkedIn URL,Company name,Company domain
0,Hasan Jilani,Director of Product Marketing,hasan.jilani@deepgram.com,https://www.linkedin.com/in/mhjilani,Deepgram,Power enterprise voice solutions with Deepgram...
1,Emily Golden,Growth Marketing Director,emily@runwayml.com,https://www.linkedin.com/in/emily-golden-b1291435,Runway,Generate images and video with AI. Text to vid...
2,Rosalie Samuels,Chief Growth Officer,rosalie.samuels@consensys.io,https://www.linkedin.com/in/rosaliesamuels,ConsenSys,Consensys is a blockchain software company bui...
3,Alex Palienko,Vice President of Growth,alex@replika.com,https://www.linkedin.com/in/alexpalienko,Replika,Replika combines a sophisticated neural networ...
4,Aurélien Rodriguez,Director of Training,aurelien@cohere.com,https://www.linkedin.com/in/aurélien-rodriguez...,Cohere,Cohere builds powerful models and AI solutions...
